# TODO：研究タイトル

**F4：パートナーシップの安定性と意思決定**

チーム名：TODO　／　メンバー：TODO

> **教育用の架空データ・架空モデル**です。Partner A・B は実在の人物ではなく、性別や役割を意味しません。満足度関数や関係解消確率の係数は架空の値で、実際の離婚率や特定の関係の将来を予測するものではありません。

## 使い方（発表前に読む・スライドには出ません）

1. 「準備」セルから順に**すべてのセルを実行**する（グラフのコードは隠してあります）
2. **TODO** をチームの内容に書き換える。**実験していない結果・考察を書かない**
3. HTML スライド：`jupyter nbconvert notebooks/06_research_presentation.ipynb --to slides --no-input --execute`

## 準備：実験環境（最初に必ず実行）

F4 リポジトリの `.py`（計算の本体）を import できるように、作業ディレクトリ（F4 のルート）を自動で探して設定します。

| 実行場所 | 準備 |
| --- | --- |
| ローカルの Jupyter / VSCode | `notebooks/` の中で開けば、そのまま動きます |
| Colab（方法A：GitHub から clone） | 下のセルの `REPO_URL` に**自分たちのリポジトリの URL** を入れる |
| Colab（方法B：ZIP をアップロード） | F4 フォルダを ZIP にして `/content` にアップロードしておく（またはセルの実行中にアップロード） |

Colab の CPU ランタイムで動きます（GPU・TPU は不要）。`textbook/`（教科書のコード）は公開リポジトリに含めない約束です。`textbook/` がないと import が失敗します。

In [ ]:
#@title 準備（F4 の .py を読み込む）
import os, sys, glob, zipfile, subprocess
from pathlib import Path

# ---- 方法A：GitHub から clone する場合だけ設定する（URL は自分たちのものを入れる）----
REPO_URL = ""                                 # 例："https://github.com/<team>/<repo>.git"（空なら clone しない）
REPO_SUBDIR = "f4-partnership-simulation"     # リポジトリの中の F4 フォルダ（ルートが F4 なら ""）

IN_COLAB = "google.colab" in sys.modules


def is_f4_root(p):
    return (Path(p) / "household_model.py").exists() and (Path(p) / "relationship_dynamics.py").exists()


def find_f4_root():
    for p in [Path.cwd(), Path.cwd().parent]:          # ローカル：notebooks/ の中、または F4 のルート
        if is_f4_root(p):
            return p.resolve()
    if not IN_COLAB:
        return None
    if REPO_URL:                                        # Colab 方法A
        dest = Path("/content/repo")
        if not dest.exists():
            subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(dest)], check=True)
        cand = dest / REPO_SUBDIR if REPO_SUBDIR else dest
        if is_f4_root(cand):
            return cand
    zips = glob.glob("/content/*.zip")                  # Colab 方法B
    if not zips:
        from google.colab import files
        print("F4 フォルダの ZIP を選んでアップロードしてください")
        zips = ["/content/" + name for name in files.upload()]
    for z in zips:
        with zipfile.ZipFile(z) as zf:
            zf.extractall("/content/f4_upload")
    for hit in glob.glob("/content/f4_upload/**/household_model.py", recursive=True):
        if is_f4_root(Path(hit).parent):
            return Path(hit).parent
    return None


F4_ROOT = find_f4_root()
if F4_ROOT is None:
    raise RuntimeError("F4 のフォルダが見つかりません。上の説明（方法A／方法B）を確認してください。")
os.chdir(F4_ROOT)
if str(F4_ROOT) not in sys.path:
    sys.path.insert(0, str(F4_ROOT))
if not (F4_ROOT / "textbook" / "max.py").exists():
    print("⚠️ textbook/ がありません。decision_strategies.py などの import が失敗します。")

# ---- 既存の .py（計算の本体）と、グラフ ----
import matplotlib.pyplot as plt
import partners as pt
import household_model as hm
import decision_strategies as ds
import relationship_dynamics as rd
import evaluation as ev
import partnership_charts as pc

pc.setup_style()                       # 文字を大きくするなら pc.setup_style(1.3)
scenarios = pt.load_scenarios()        # シナリオ A〜D（すべて架空）
params = rd.load_parameters()          # モデルのパラメータ（すべて架空）
events = pt.load_stress_events()       # ストレスイベントの組
SC = pt.scenario_by_id(scenarios, "A") # 基準にする初期条件（Scenario A）
P, HH = SC["partners"], SC["household"]
COND0 = dict(HH, stress=params["base_stress"])   # ストレスイベントのない年の生活条件
CANDS = ds.candidate_grid(params["share_step"])  # 分担の候補（全戦略で共通）
SEEDS = range(5)                       # 乱数シード
REPLICATES = 200                       # シードごとの反復回数
YEARS = params["simulation_years"]

SAVE_FIGURES = False                   # True にすると、グラフを notebooks/figures/ に PNG で保存する
FIG_DIR = "notebooks/figures"


def show(fig, name):
    """グラフを表示し、SAVE_FIGURES = True なら PNG も保存する"""
    if SAVE_FIGURES:
        print("saved:", pc.save(fig, f"{FIG_DIR}/{name}.png"))
    plt.show()


def table(rows, columns):
    """辞書のリストを表で表示する。columns = [(キー, 見出し, 書式)]。pandas があれば DataFrame で表示"""
    data = [{h: ("-" if r.get(k) is None else f.format(r[k])) for k, h, f in columns} for r in rows]
    try:
        import pandas as pd
        from IPython.display import display
        display(pd.DataFrame(data))
    except ImportError:
        print(ev.format_table(rows, columns))


print("F4_ROOT:", F4_ROOT)
print("シナリオ:", [s["label"] for s in scenarios])
print("モデルのパラメータ（架空）:", {k: params[k] for k in ("alpha", "baseline_hazard", "beta", "gamma",
                                                       "base_stress", "fairness_lambda", "share_step", "simulation_years")})

## 研究背景

- TODO：なぜ家事分担・経済的負担・コミュニケーションと関係の継続を、アルゴリズムで考えるのか
- TODO：F3（出会いとマッチング）との関係

## Research Question

- TODO：チームが選んだ RQ（RQ1〜RQ6 から。例：RQ6「短期的な満足度最大化と長期的な関係継続は一致するか」）

## 仮説

- TODO：**実験の前に**立てた仮説を書く

## パートナーの意思決定モデル

- 2 人（Partner A・B）がそれぞれ意思決定主体。A・B は区別のための名前で、性別や役割を意味しない
- 各パートナー：所得・労働時間・家事に使える時間・希望の分担・コミュニケーション・公平性の重視度・満足度
- 毎年、分担 $(x, y)$ を決める → 2 人の満足度が変わる → 関係の状態が変わる

In [ ]:
#@title パートナー（架空）
table([P[pid] for pid in pt.PARTNERS],
      [("id", "partner", "{}"), ("annual_income", "income", "{}"), ("weekly_work_hours", "work h", "{}"),
       ("available_housework_hours", "housework limit h", "{}"), ("preferred_housework_share", "wants housework", "{:.0%}"),
       ("preferred_financial_share", "wants expenses", "{:.0%}"), ("communication_score", "communication", "{}")])

## 家事・経済的負担モデル

- 家事：$\text{Housework}_A = x \times \text{Total}$、生活費：$\text{Financial}_A = y \times \text{Expenses}$（B は残り）
- $x$ と $y$ は別々に決める。家事に使える時間・所得を超える分担は実行不可能

In [ ]:
#@title 家事時間
hw_rows = []
for name, x in {"50:50": 0.5, "work hours": hm.work_hours_housework_share(P), "80:20": 0.8,
                "preference": hm.preference_housework_share(P)}.items():
    a, b = hm.housework_hours(x, HH["total_housework_hours"])
    hw_rows.append({"name": name, "A": a, "B": b, "limit_A": P["A"]["available_housework_hours"],
                    "limit_B": P["B"]["available_housework_hours"]})
show(pc.plot_hours_bars(hw_rows), "06_housework_hours")

## 満足度モデル

- 満足度 = 基準値 − 希望とのずれ + コミュニケーション − ストレス + 自由時間 − 超過 − 不公平感 − 家計の圧迫（0〜1）
- 平均満足度・最低満足度・満足度の差を区別する。**係数は架空の値**

In [ ]:
#@title 分担率と満足度
xs = ds.share_grid(0.05)
show(pc.plot_share_sweep(xs, [hm.utility("A", x, 0.5, P, COND0, params) for x in xs],
                         [hm.utility("B", x, 0.5, P, COND0, params) for x in xs],
                         "x = Partner A's housework share (expenses 50:50)", "Housework share and satisfaction"),
     "06_sweep")

## 意思決定戦略

| 戦略 | 選び方 |
| --- | --- |
| Fixed | 分担を固定 |
| Greedy | その年の平均満足度が最大（全探索、$O(KL)$） |
| Fairness-aware | 平均 − λ × 差 が最大 |
| Adaptive | 定期的に見直し、近くの候補へ少しずつ |

同じ候補集合・同じ制約・同じ満足度関数を使う

## 関係状態の時間変化

- $S_i(t) = (1-\alpha) S_i(t-1) + \alpha U_i(t)$、$R(t)$ = 平均と最低の加重平均
- $p(t) = \text{baseline} + \beta (1 - R(t)) + \gamma \cdot \text{Stress}(t)$（0〜1）、$u < p(t)$ なら解消
- **架空のモデル**。実際の離婚確率を表さない

## 実験条件

- TODO：初期条件・ストレスイベント・シード数・反復回数・変えたパラメータをチームの実験に合わせて書く

In [ ]:
#@title 実験条件
EVENTS = events["combined"]          # TODO：チームの実験条件にする
EV_YEARS = rd.event_years(EVENTS)
res = ev.compare_strategies(SC, params, EVENTS, ds.STRATEGIES, REPLICATES, SEEDS)
print(f"初期条件 Scenario {SC['id']}、シード {len(SEEDS)} × 反復 {REPLICATES} 回、イベント:",
      "、".join(f"Year {e['year']} {e['note']}" for e in EVENTS))

## 結果：満足度と公平性

In [ ]:
#@title 満足度と公平性
show(pc.plot_strategy_metrics(res, [("mean_satisfaction", "mean satisfaction"), ("min_satisfaction", "min satisfaction"),
                                    ("fairness", "fairness index")], "Satisfaction and fairness (20-year average, model)"),
     "06_metrics")

## 結果：関係継続率

In [ ]:
#@title 継続率の曲線
show(pc.plot_survival({s: res[s]["survival"] for s in ds.STRATEGIES}, "Continuation rate by strategy (model)"),
     "06_survival")

In [ ]:
#@title 5・10・20 年継続率
show(pc.plot_checkpoints(res, "5/10/20-year continuation (model, mean ± sd over seeds)"), "06_checkpoints")

## 結果：短期の満足度と長期の継続

In [ ]:
#@title 短期と長期
show(pc.plot_short_vs_long(res, "Short-term satisfaction vs long-term continuation (model)"), "06_short_vs_long")

## 結果：チームの追加実験

- TODO：チームの実験（ストレス条件・コミュニケーション・感度分析など）のグラフに置き換える

In [ ]:
#@title チームの追加実験
# TODO：チームの追加実験に置き換える（例：ストレス条件別の 10 年継続率）
rows = []
for name in ["none", "income_drop", "housework_increase", "combined"]:
    rows.append({"name": name.replace("_", "\n"),
                 **ev.compare_strategies(SC, params, events[name], ds.STRATEGIES, REPLICATES, SEEDS)})
show(pc.plot_condition_bars(rows, "continuation_10", "10-year continuation by stress condition (model)",
                            "10-year continuation rate"), "06_team_experiment")

## 考察

- TODO：仮説は支持されたか。**どの仮定の下で**そう言えるか
- TODO：差はシード間のばらつきより大きいか

## モデルの限界

- 教育用の**架空のシミュレーション**。実際の離婚確率や関係の将来を予測しない
- 満足度関数・解消確率の係数は架空。家事分担と関係継続の関係は**モデルの仮定**に依存する
- 50:50 が常に公平とは限らない。何を公平とみなすかはモデル上の選択
- 所得の多寡で人の価値を評価しない。性別による役割を固定しない
- 関係の解消を個人の失敗として扱わない。関係の継続が常に望ましいとは限らない
- 強制・暴力・安全上の問題は、満足度の最適化の問題ではない
- TODO：チームが気づいた限界を追加する

## 結論

- TODO：RQ への答えを、**仮定とともに**書く

## 今後の課題

- TODO：Advanced A〜F のどれに取り組むか（繰り返しゲーム、強化学習、動的ストレス、多目的最適化、F3 との連携、意思決定の非対称性）

## 参考文献

- 柴田望洋『新・明解 Pythonで学ぶアルゴリズムとデータ構造（第2版）』SBクリエイティブ（第2章 最大値、第3章 線形探索、第6章 単純挿入ソート）
- F4 リポジトリの `README.md`（モデルの仮定・計算量・モデルの限界と倫理的配慮）
- データ：`data/sample_partnerships.json`（この教材のための**架空のデータ**）